# LeVJEPA training workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLO-lab/LeVJEPA/blob/main/notebooks/training_workshop.ipynb)

This notebook walks through **training LeVJEPA from scratch, end to end, inside the
notebook** — a miniature version of the full recipe in
[`main.py`](https://github.com/MLO-lab/LeVJEPA/blob/main/main.py), built
from the same components
([`module.py`](https://github.com/MLO-lab/LeVJEPA/blob/main/module.py) for the
model and loss,
[`data/loader.py`](https://github.com/MLO-lab/LeVJEPA/blob/main/data/loader.py)
for the multi-crop transform). You will:

1. grab a few minutes of a **Walking Tours** video as toy data,
2. build the multi-crop views (one global + several local crops of the same clip),
3. assemble the model — a tiny ViT with RoPE, block-causal attention, and 90% token
   drop — plus the projector and the SIGReg loss,
4. step through the training loop — already written, but decomposed into four small
   functions so each piece of the objective is readable on its own — and train for a
   thousand steps,
5. check what happened: the embedding distribution SIGReg shapes, and how the patch
   tokens change between an untrained and a trained encoder.

It runs on a CUDA GPU (a free Colab T4 is plenty), an Apple Silicon Mac (MPS), or
plain CPU — training takes a few minutes on a GPU; on MPS or CPU lower `N_STEPS` if
you're impatient. There is no Lightning, no hydra, and no cluster here — the point is
to see that the entire method is one encoder, one projector, and one loss.

Setup — on **Colab**, pick a GPU runtime (*Runtime → Change runtime type → T4 GPU*)
and just run the next cell: it clones the repo and installs the few extra
dependencies. **Locally** the next cell is a no-op; from the repo root:

```bash
uv sync --extra notebook
uv run jupyter lab notebooks/training_workshop.ipynb
```

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    import subprocess
    from pathlib import Path

    if not Path("/content/levjepa").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/MLO-lab/LeVJEPA.git", "/content/levjepa"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "einops", "av", "pylance"],
        check=True,
    )
    os.chdir("/content/levjepa/notebooks")
    print("Colab setup done: repo cloned, dependencies installed")

In [ ]:
import copy
import math
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, "..")  # repo root, so we can import module.py / data/

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from einops import rearrange
from PIL import Image

from module import SIGReg, Projector, vit_tiny
from data.loader import VJEPAMultiCropTransform

torch.manual_seed(0)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"running on {device}")

## 1. Toy data: five minutes of Venice

Walking Tours (Venkataramanan et al., *Is ImageNet worth 1 video?*, ICLR 2024) is ten
hours-long continuous first-person city walks — unlabeled, uncurated, and surprisingly
effective: the paper's consumer-hardware experiment pretrains a ViT-Tiny on eight of
these videos for 12 hours on a single RTX 5080 and reaches 25.2% frozen ImageNet
accuracy. We use the same source, just much less of it: a **five-minute slice of the
Venice walk at 360p** instead of the full 25 GB dataset.

On **Colab** the cell below simply downloads the decoded frames, prebuilt, from the
[Hugging Face Hub](https://huggingface.co/datasets/galilai-group/levjepa-workshop)
(~300 MB — streaming from YouTube is unreliable on datacenter IPs). **Locally** the
same frames are produced from the video itself, with no ffmpeg install and no 25 GB
download: yt-dlp only *resolves* the direct stream URL, and PyAV — whose wheel bundles
its own ffmpeg libraries on every platform — seeks to the 20-minute mark and decodes
five minutes straight off the stream (a minute or two, network permitting). Either
way the frames are cached to `workshop_frames.pt`, so this cell is slow only once. If
you already ran `scripts/download_walking_tours.sh`, the local Venice file is used
instead — and you can always point `VIDEO_PATH` at any video file you have lying
around.

In [ ]:
VIDEO_PATH = None       # set to any local video file to use it instead
FRAMES_CACHE = Path("workshop_frames.pt")

TARGET_FPS = 7.5        # the training sampling rate
SHORT_SIDE = 160        # decoded frame resolution
START_SEC = 1200        # skip the first 20 minutes of the walk
MAX_FRAMES = 2250       # 5 minutes at 7.5 fps -> ~300 MB in memory


def _resize_chunks(frames, short_side=SHORT_SIDE):
    """(N, 3, H, W) uint8 -> short side `short_side`, resized in chunks."""
    scale = short_side / min(frames.shape[-2:])
    out = []
    for chunk in frames.split(256):
        out.append(
            F.interpolate(
                chunk.float(), scale_factor=scale, mode="bilinear", antialias=True
            ).round_().clamp_(0, 255).to(torch.uint8)
        )
    return torch.cat(out)


def decode_video(source, start_sec=0.0, target_fps=TARGET_FPS, max_frames=MAX_FRAMES):
    """Local file or https URL -> (N, 3, H, W) uint8 frames at ~`target_fps`."""
    import av

    with av.open(str(source)) as container:
        stream = container.streams.video[0]
        step = max(1, round(float(stream.average_rate) / target_fps))
        if start_sec:
            container.seek(int(start_sec / stream.time_base), stream=stream)
        decoded = []
        for i, frame in enumerate(container.decode(stream)):
            if i % step == 0:
                decoded.append(torch.from_numpy(frame.to_ndarray(format="rgb24")))
            if len(decoded) >= max_frames:
                break
    return _resize_chunks(torch.stack(decoded).permute(0, 3, 1, 2))


if FRAMES_CACHE.exists():
    frames = torch.load(FRAMES_CACHE)
    print(f"loaded cached frames from {FRAMES_CACHE}")
elif VIDEO_PATH is not None:
    frames = decode_video(VIDEO_PATH)
elif IN_COLAB:
    from huggingface_hub import hf_hub_download

    print("downloading prebuilt frame cache from the Hugging Face Hub ...")
    hf_hub_download(
        "galilai-group/levjepa-workshop", FRAMES_CACHE.name,
        repo_type="dataset", local_dir=".",
    )
    frames = torch.load(FRAMES_CACHE)
else:
    local = sorted(Path("../data/walking_tours/videos").glob("Venice.mp4"))
    if local:
        print(f"decoding already-downloaded {local[0]}")
        frames = decode_video(local[0], start_sec=START_SEC)
    else:
        print("resolving stream url ...")
        url = subprocess.run(
            [
                sys.executable, "-m", "yt_dlp", "-g",
                "--format",
                "bestvideo[height<=360][ext=mp4][protocol=https]"
                "/bestvideo[height<=360][ext=mp4]/bestvideo[height<=360]",
                "https://www.youtube.com/watch?v=fGX0Te6pFvk",  # Venice
            ],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        print("decoding 5 minutes off the stream ...")
        frames = decode_video(url, start_sec=START_SEC)

if not FRAMES_CACHE.exists():
    torch.save(frames, FRAMES_CACHE)
print(f"{frames.shape[0]} frames of {tuple(frames.shape[2:])}, "
      f"{frames.numel() / 1e6:.0f} MB in memory")

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, t in zip(axes, np.linspace(0, len(frames) - 1, 6, dtype=int)):
    ax.imshow(frames[t].permute(1, 2, 0).numpy())
    ax.set_title(f"frame {t}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Views: one global, several locals

Each training sample is a short clip turned into **one global view** and **V local
views**. All views share the *same temporal window* — they differ only spatially and
photometrically: the global view is a near-full-frame crop left photometrically clean
(it's the prediction target), the locals are small aggressive crops with color jitter,
grayscale, and flips. The full recipe uses 224px globals and 96px locals; we shrink to
**112 / 48** so the token grids are 7x7 and 3x3 per frame and everything fits a small
budget, with **8 frames** per clip instead of 16.

We reuse `VJEPAMultiCropTransform` — the exact class the real training pipeline uses —
and wrap the in-memory frame tensor in a five-line dataset that samples a random
temporal window.

In [ ]:
NUM_FRAMES = 8      # frames per clip (16 in the full recipe)
FRAME_STRIDE = 2    # stored frames are ~7.5 fps, so stride 2 spans ~2 s
GLOBAL_SIZE = 112   # 7x7 patch grid  (224 in the full recipe)
LOCAL_SIZE = 48     # 3x3 patch grid  (96 in the full recipe)
NUM_LOCALS = 4

transform = VJEPAMultiCropTransform(
    global_size=GLOBAL_SIZE,
    local_size=LOCAL_SIZE,
    local_crops_number=NUM_LOCALS,
)


class ToyClipDataset(torch.utils.data.Dataset):
    """Random `NUM_FRAMES`-frame windows out of one in-memory video."""

    def __init__(self, frames, clips_per_epoch=2048):
        self.frames = frames
        self.span = NUM_FRAMES * FRAME_STRIDE
        self.clips_per_epoch = clips_per_epoch

    def __len__(self):
        return self.clips_per_epoch

    def __getitem__(self, idx):
        start = torch.randint(0, len(self.frames) - self.span, (1,)).item()
        clip = self.frames[start : start + self.span : FRAME_STRIDE]  # (T, 3, H, W)
        return transform({"frame": clip})


BATCH_SIZE = 128
loader = torch.utils.data.DataLoader(
    ToyClipDataset(frames), batch_size=BATCH_SIZE, num_workers=0
)

batch = next(iter(loader))
print("global_frame:", tuple(batch["global_frame"].shape), "(B, T, C, H, W)")
print("local_frames:", tuple(batch["local_frames"].shape), "(B, V, T, C, H, W)")

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225])


def denormalize(x):
    x = x.cpu() * IMAGENET_STD[:, None, None] + IMAGENET_MEAN[:, None, None]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()


# One sample: the global view's frames on top, the local views below.
fig, axes = plt.subplots(2, NUM_FRAMES, figsize=(14, 4))
for t in range(NUM_FRAMES):
    axes[0, t].imshow(denormalize(batch["global_frame"][0, t]))
    axes[0, t].set_title(f"global t={t}", fontsize=8)
for v in range(NUM_LOCALS):
    axes[1, v].imshow(denormalize(batch["local_frames"][0, v, 0]))
    axes[1, v].set_title(f"local {v} (t=0)", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. The model: an encoder, a projector, and a loss

That's the complete trainable architecture — no target encoder, no predictor, no
stop-gradient anywhere. Every constructor argument below is one of the paper's design
choices:

- `tubelet_size=1` — one token per frame per patch; no temporal aggregation at the
  input (Section "Temporal patch aggregation is not required").
- `use_rope=True` — factorized 3D rotary embeddings encode *relative* position, which
  is what lets the same encoder digest both 112px globals and 48px locals without
  interpolating anything.
- `token_drop_rate=0.9` — during training the encoder keeps a random 10% of each
  view's patch tokens. This is the augmentation-not-approximation finding: it cuts the
  step cost ~10x *and* improves the learned representation. It is automatically inert
  under `.eval()`.
- `attn_mode="block_causal"` — bidirectional within a frame, causal across frames,
  so every frame's representation depends on the past only. In the paper this costs
  nothing in accuracy and is the default.

The projector maps the `[cls]` token into the small space where the loss lives (it's
needed because the encoder's final LayerNorm confines `[cls]` to a sphere, where an
isotropic Gaussian can't be optimized), and `SIGReg` is the regularizer that provably
excludes collapse. The loss weight **λ = 0.02** is the objective's only hyperparameter
— the paper never tunes it, and neither do we.

In [ ]:
encoder = vit_tiny(
    img_size=GLOBAL_SIZE,
    patch_size=16,
    num_frames=NUM_FRAMES,
    tubelet_size=1,
    use_rope=True,
    token_drop_rate=0.9,
    attn_mode="block_causal",
).to(device)
projector = Projector(encoder.embed_dim, hidden_dim=1024, output_dim=128).to(device)
sigreg = SIGReg().to(device)
SIGREG_WEIGHT = 0.02

# Frozen untrained copy, for the before/after comparison in section 5.
encoder_untrained = copy.deepcopy(encoder).eval()

n_params = sum(p.numel() for p in encoder.parameters()) + sum(
    p.numel() for p in projector.parameters()
)
print(f"{n_params / 1e6:.1f}M trainable parameters "
      f"({sum(p.numel() for p in encoder.parameters()) / 1e6:.1f}M encoder)")

## 4. The training step, piece by piece

The whole objective, exactly as in `main.py`'s `multiview_forward`:

$$\mathcal{L} = \underbrace{\tfrac{1}{V+1}\sum_v \lVert z_0 - z_v \rVert^2}_{\text{invariance}} \; + \; \lambda \, \mathcal{L}_{\text{SIGReg}}$$

where $z_0$ is the projected `[cls]` embedding of the global view and $z_v$ those of
the locals. The loop below is already written — the next four cells just take it apart
into its four moves so you can read each one in isolation before running them
together.

**Move 1 — encode.** Both view types go through the *same* encoder. The locals are
folded into the batch dimension for one batched forward, and from each view we keep
only the `[cls]` token — the clip-level readout. With the 90% token drop active, the
global view is ~40 tokens and each local ~7, which is why this runs in a notebook.

In [ ]:
def encode_views(batch):
    """Run the encoder over all views, keep each view's [cls] token.

    Returns (global_cls, local_cls) of shapes (B, 1, D) and (B, V, D).
    """
    global_frame = batch["global_frame"].to(device)   # (B, T, C, H, W)
    local_frames = batch["local_frames"].to(device)   # (B, V, T, C, H, W)
    B = global_frame.shape[0]

    global_tokens = encoder(rearrange(global_frame, "b t c h w -> b c t h w"))
    global_cls = global_tokens[:, 0].unsqueeze(1)

    local_tokens = encoder(rearrange(local_frames, "b v t c h w -> (b v) c t h w"))
    local_cls = rearrange(local_tokens[:, 0], "(b v) d -> b v d", b=B)
    return global_cls, local_cls

**Move 2 — project.** The projector maps every `[cls]` token into the K-dimensional
space where the loss lives. One call handles all views at once; the global view stays
at index 0.

In [ ]:
def project_views(global_cls, local_cls):
    """All [cls] tokens -> loss-space embeddings, shape (B, V+1, K)."""
    return projector(torch.cat([global_cls, local_cls], dim=1))

**Move 3 — the two losses.** The invariance term pulls every view's embedding toward
the global view's. Look closely at what it *doesn't* do: `emb[:, :1]` — the target —
is not detached. Gradients flow through both sides; there is no stop-gradient and no
EMA teacher. Alone, this loss has a trivial minimum (map everything to one constant).
SIGReg is what forbids it: it projects the batch of embeddings onto random directions
and penalizes any deviation from a standard Gaussian — and a collapsed batch, having
zero variance, is maximally non-Gaussian.

In [ ]:
def invariance_loss(emb):
    """MSE between the global embedding (index 0) and every view, target included."""
    return (emb[:, :1] - emb).pow(2).mean()


def regularization_loss(emb):
    """SIGReg over each view's batch of embeddings: (B, V+1, K) -> (V+1, B, K)."""
    return sigreg(rearrange(emb, "b v d -> v b d"))

**Move 4 — put it together.** One step is: encode, project, add the two losses with
λ = 0.02, backprop, done. `train()` just repeats that and records the two losses.

In [ ]:
def training_step(batch):
    emb = project_views(*encode_views(batch))
    return invariance_loss(emb), regularization_loss(emb)


def train(n_steps, lr=1e-4):
    optimizer = torch.optim.AdamW(
        list(encoder.parameters()) + list(projector.parameters()),
        lr=lr, weight_decay=0.04,
    )
    encoder.train(), projector.train()
    history = {"pred": [], "sigreg": []}
    step = 0
    while step < n_steps:
        for batch in loader:
            pred_loss, sigreg_loss = training_step(batch)
            loss = pred_loss + SIGREG_WEIGHT * sigreg_loss
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            history["pred"].append(pred_loss.item())
            history["sigreg"].append(sigreg_loss.item())
            step += 1
            if step % 50 == 0:
                print(f"step {step:4d}  pred {pred_loss.item():.4f}  "
                      f"sigreg {sigreg_loss.item():.3f}")
            if step >= n_steps:
                break
    return history


N_STEPS = 1000  # a few minutes on GPU; lower this on MPS or CPU
history = train(N_STEPS)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history["pred"])
axes[0].set_title("invariance loss (local -> global MSE)")
axes[1].plot(history["sigreg"], color="tab:orange")
axes[1].set_title("SIGReg loss")
for ax in axes:
    ax.set_xlabel("step")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Watch the interplay: the invariance loss *cannot* just fall to zero by collapsing
everything to one point, because SIGReg penalizes exactly that. Instead the two settle
into a balance — embeddings that are stable across views of the same clip, yet spread
like a Gaussian across clips.

## 5. What did it learn?

**First check: did SIGReg do its job?** The regularizer claims the projected
embeddings should look like $\mathcal{N}(0, I)$ along *any* direction. We can test
that directly: embed a few batches, project onto random unit vectors, and compare the
histogram against a standard normal pdf. A collapsed encoder would put all its mass at
a single point.

In [ ]:
encoder.eval(), projector.eval()  # eval() also disables token drop

with torch.no_grad():
    embs = []
    it = iter(loader)
    for _ in range(4):
        b = next(it)
        g = encoder(rearrange(b["global_frame"].to(device), "b t c h w -> b c t h w"))
        embs.append(projector(g[:, 0]).cpu())
    embs = torch.cat(embs)  # (4B, K)

dirs = F.normalize(torch.randn(3, embs.shape[-1]), dim=-1)
proj = embs @ dirs.T  # (N, 3)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
t = np.linspace(-4, 4, 200)
for i, ax in enumerate(axes):
    ax.hist(proj[:, i].numpy(), bins=30, density=True, alpha=0.7)
    ax.plot(t, np.exp(-t**2 / 2) / math.sqrt(2 * math.pi), "k--", label="N(0,1)")
    ax.set_title(f"random projection {i}", fontsize=10)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

With only a thousand steps and 512 embeddings the histograms are rough, but they
should already be unimodal, centered, and roughly unit-scale — nothing like the single
spike a collapsed solution would produce.

**Second check: what happened to the patch tokens?** They never receive any loss —
only `[cls]` is supervised — yet in the paper semantically organized patch
representations emerge anyway. Two choices make this visible even for a tiny model on
five minutes of data. First, the probe input: the bundled whippet photo
(`data/dog.jpg`, the same image the feature-visualization notebook uses), repeated
along the temporal axis — the paper's protocol for static images — so there's one
clear foreground object to separate from couch and background. Second, the
resolution: because RoPE encodes *relative* positions, the encoder trained at 112px
can be probed at **224px** — a 14x14 grid, 196 tokens per frame instead of 49 — with
no interpolation of anything. Comparing the same input through the untrained and the
trained encoder shows the tokens starting to organize.

In [ ]:
def pca_rgb(feat, grid):
    feat = feat - feat.mean(dim=0)
    _, _, v = torch.pca_lowrank(feat, q=3)
    comps = (feat @ v).numpy()
    lo, hi = np.percentile(comps, [2, 98], axis=0)
    return np.clip((comps - lo) / (hi - lo + 1e-8), 0, 1).reshape(grid, grid, 3)


# The bundled whippet image, repeated along the temporal axis and probed at
# double the training resolution -- RoPE makes the grid size a free choice.
EVAL_SIZE = 224     # 14x14 patch grid

image = Image.open("data/dog.jpg").convert("RGB")
w, h = image.size
scale = EVAL_SIZE / min(w, h)
image = image.resize((round(w * scale), round(h * scale)), Image.BICUBIC)
w, h = image.size
left, top = (w - EVAL_SIZE) // 2, (h - EVAL_SIZE) // 2
image = image.crop((left, top, left + EVAL_SIZE, top + EVAL_SIZE))
img_t = torch.from_numpy(np.array(image)).float().div_(255).permute(2, 0, 1)
img_t = (img_t - IMAGENET_MEAN[:, None, None]) / IMAGENET_STD[:, None, None]
clip = img_t[None, :, None].repeat(1, 1, NUM_FRAMES, 1, 1).to(device)

GRID = EVAL_SIZE // 16
with torch.no_grad():
    feats = {
        "untrained": encoder_untrained(clip),
        "trained": encoder(clip),
    }

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(denormalize(img_t))
axes[0].set_title("input")
for ax, (name, tok) in zip(axes[1:], feats.items()):
    grid_tokens = tok[0, 1:].reshape(NUM_FRAMES, GRID, GRID, -1)[-1]  # last slot
    ax.imshow(pca_rgb(grid_tokens.reshape(GRID * GRID, -1).cpu(), GRID),
              interpolation="nearest")
    ax.set_title(f"patch-token PCA, {name}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Where to go from here

Everything you just ran *is* the method — the full recipe differs only in scale:

- **The real training run** is `sbatch slurm/train_walking_tours_vitb.slurm` (or
  `uv run python main.py` locally): ViT-B/16, 224px/96px views, 16 frames, 95% token
  drop, batch 3072, driven by
  [`conf/config.yaml`](https://github.com/MLO-lab/LeVJEPA/blob/main/conf/config.yaml).
  The forward pass is the same four moves you stepped through in section 4.
- **The paper's consumer-hardware run** is this notebook taken seriously: a ViT-Tiny
  on eight Walking Tours videos for 12 hours on one 16 GB GPU reaches 25.2% frozen
  ImageNet accuracy. `bash scripts/download_walking_tours.sh` gets you the data.
- **The release checkpoint** — a ViT-L trained on 1.8M clips — is explored in
  [feature_visualization.ipynb](https://github.com/MLO-lab/LeVJEPA/blob/main/notebooks/feature_visualization.ipynb),
  where the emergent patch-token structure you glimpsed above becomes sharp object
  segmentations.